# Doge Defenders — M3 exploratory analysis

**Course:** BUA 3336 · **Team:** Doge Defenders  
**Question:** For one coin on one day, was the price higher about 90 days later?

This notebook reads `daily_reports.csv` and describes the file before any model is fit. It does not place trades. On Google Colab, use **Runtime → Restart and run all**.


## 1. Open the project file

The memo has to quote this file, not a remembered summary. On Colab we clone the project branch that contains `daily_reports.csv`. On a laptop we use the copy already in the folder. We then keep only the rows where the 90-day result is already known. The 15 rows for September 26, 2026 are still in the future, so they cannot tell us which coins rose.


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/The0gCarrot/doge-defenders.git"
BRANCH = "m3-exploratory"

if "google.colab" in sys.modules:
    dest = Path("/content/doge-defenders")
    if not dest.exists():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(dest)]
        )
    import os
    os.chdir(dest)

def find_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        csv_path = candidate / "data" / "processed" / "daily_reports.csv"
        if csv_path.exists() and (candidate / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("daily_reports.csv was not found. Clone the project branch first.")

REPO = find_repo()
CSV_PATH = REPO / "data" / "processed" / "daily_reports.csv"
FIG_DIR = REPO / "docs" / "m3_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(CSV_PATH)
labeled = raw[raw["positive_90d_return"].isin([0, 1])].copy()
labeled["positive_90d_return"] = labeled["positive_90d_return"].astype(int)
labeled["report_day"] = pd.to_datetime(labeled["report_date"].str.slice(0, 10))

print("File:", CSV_PATH)
print("Rows in the file:", len(raw))
print("Columns:", raw.shape[1])
print("Rows with a known 90-day result:", len(labeled))
print("Rows still in the future (left out of the comparisons below):", int((raw["positive_90d_return"] == -999).sum()))
n_up = int((labeled["positive_90d_return"] == 1).sum())
n_down = int((labeled["positive_90d_return"] == 0).sum())
print("Up (1):", n_up, f"({n_up / len(labeled):.1%})")
print("Not up (0):", n_down, f"({n_down / len(labeled):.1%})")


## 2. One variable at a time

A manager who hears an average can picture the wrong coin. Price, dollar volume, and market value run from meme coins to Bitcoin, so a few huge rows can lift the average far above the middle row. We put the mean next to the median for the six numbers that matter most to the question. Where those two are far apart, the average is describing a rare row, not a typical day. The 24-hour return is included on purpose: if the mean and the median agree, the average is safe to quote.


In [ ]:
KEY_NUMBERS = [
    "price_usd",
    "volume_24h_usd",
    "market_cap_usd",
    "momentum_30d_pct",
    "news_count_10d",
    "return_24h",
]

rows = []
for column in KEY_NUMBERS:
    series = pd.to_numeric(labeled[column], errors="coerce")
    rows.append(
        {
            "variable": column,
            "mean": series.mean(),
            "median": series.median(),
            "skew": series.skew(),
            "minimum": series.min(),
            "maximum": series.max(),
        }
    )
univariate = pd.DataFrame(rows)
print(univariate.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

price = pd.to_numeric(labeled["price_usd"])
btc_eth_price_share = price[labeled["token_symbol"].isin(["BTC", "ETH"])].sum() / price.sum()
mcap = pd.to_numeric(labeled["market_cap_usd"])
btc_mcap_share = mcap[labeled["token_symbol"].eq("BTC")].sum() / mcap.sum()
zero_news_share = (pd.to_numeric(labeled["news_count_10d"]) == 0).mean()
print(f"Bitcoin plus Ethereum share of the summed prices: {btc_eth_price_share:.1%}")
print(f"Bitcoin share of the summed market values: {btc_mcap_share:.1%}")
print(f"Share of labeled rows with zero headlines: {zero_news_share:.1%}")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
log_mcap = np.log10(mcap)
ax.hist(log_mcap, bins=18, color="#1f4e79", edgecolor="white")
ax.axvline(np.log10(mcap.mean()), color="#b85c38", linewidth=2, label=f"Mean ${mcap.mean()/1e9:,.1f} billion")
ax.axvline(np.log10(mcap.median()), color="#c4a35a", linewidth=2, label=f"Median ${mcap.median()/1e9:,.2f} billion")
ax.set_xlabel("Market value, log scale (10 = $10 billion)")
ax.set_ylabel("Number of coin-days")
ax.set_title("Most coin-days are small. The average is not.")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_market_cap_distribution.png", dpi=140, bbox_inches="tight")
plt.show()


## 3. Each predictor against the 90-day result

A column can look important and still not separate the coins that rose from the coins that did not. For each numeric predictor we compare the median on days that finished up with the median on days that did not. For category and for headline opinion we compare the share that finished up. The 24-hour return is the check we expect to be weak: one day of price change should not, by itself, tell us the next 90 days. The chart asks a sharper version of the same question: if we sort days by the prior 30-day price change, does the share that finished higher actually change?


In [ ]:
up = labeled["positive_90d_return"].eq(1)
down = labeled["positive_90d_return"].eq(0)
compare_columns = [
    "momentum_30d_pct",
    "drawdown_from_peak_pct",
    "return_24h",
    "market_cap_usd",
    "news_count_10d",
    "news_delta",
]
class_rows = []
for column in compare_columns:
    series = pd.to_numeric(labeled[column], errors="coerce")
    class_rows.append(
        {
            "predictor": column,
            "median_when_up": series[up].median(),
            "median_when_not_up": series[down].median(),
            "correlation_with_target": series.corr(labeled["positive_90d_return"]),
        }
    )
print(pd.DataFrame(class_rows).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

def up_rate_table(frame, column):
    grouped = (
        frame.groupby(column, observed=True)["positive_90d_return"]
        .agg(up_rate="mean", rows="count", up_count="sum")
        .reset_index()
        .sort_values("up_rate", ascending=False)
    )
    return grouped

print("\nCategory")
print(up_rate_table(labeled, "category").to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print("\nHeadline opinion")
print(up_rate_table(labeled, "press_opinion").to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

labeled["has_headline"] = labeled["press_opinion"].ne("none")
print("\nAny headline versus none")
print(up_rate_table(labeled, "has_headline").to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

momentum = pd.to_numeric(labeled["momentum_30d_pct"])
labeled["momentum_quartile"] = pd.qcut(momentum, 4, duplicates="drop")
quartile = up_rate_table(labeled, "momentum_quartile")
print("\n30-day momentum quartiles (lowest momentum first)")
print(quartile.sort_values("momentum_quartile").to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

fig, ax = plt.subplots(figsize=(7.2, 3.2))
quartile = quartile.sort_values("momentum_quartile")
labels = [
    "Worst month\n(fell the most)",
    "Second",
    "Third",
    "Best month\n(fell least or rose)",
]
bars = ax.bar(labels, quartile["up_rate"] * 100, color=["#1f4e79", "#4f7ea8", "#8fadcc", "#b85c38"])
for bar, rate, n in zip(bars, quartile["up_rate"], quartile["rows"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.2, f"{rate:.0%}\n({int(n)} rows)", ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, 115)
ax.set_ylabel("Share higher 90 days later")
ax.set_title("A hot 30-day run was the weaker group")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_momentum_vs_target.png", dpi=140, bbox_inches="tight")
plt.show()

by_day = labeled.groupby("report_day")["positive_90d_return"].mean().sort_index()
print("\nDaily share higher 90 days later")
print(f"Lowest day: {by_day.idxmin().date()} {by_day.min():.1%}")
print(f"Days at 100%: {int((by_day == 1).sum())} of {len(by_day)}. First of those days: {by_day[by_day == 1].index.min().date()}")
mid = pd.Timestamp("2026-06-10")
early = labeled.loc[labeled["report_day"] < mid, "positive_90d_return"]
late = labeled.loc[labeled["report_day"] >= mid, "positive_90d_return"]
print(f"Before {mid.date()}: {int(early.sum())} of {len(early)} rows higher ({early.mean():.1%})")
print(f"On or after {mid.date()}: {int(late.sum())} of {len(late)} rows higher ({late.mean():.1%})")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(by_day.index, by_day.values * 100, color="#1f4e79", marker="o", markersize=3.5)
ax.axvline(mid, color="#b85c38", linewidth=1.2, linestyle="--")
ax.set_ylabel("Share of the 15 coins that were higher")
ax.set_xlabel("Report date")
ax.set_ylim(0, 105)
ax.set_title("Later start dates almost all finished higher")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "03_up_rate_by_date.png", dpi=140, bbox_inches="tight")
plt.show()


## 4. Which numeric columns move together

Before we trust two columns as two separate facts, we check whether they are really the same fact written twice. The heatmap uses correlation, which only notices a straight-line relationship between two numbers. We leave out the 90-day result, the model scores, and the practice-portfolio columns. Those either are the answer or were built from the answer. We also leave out category, exchange, and headlines, because correlation cannot take a name or a sentence as input.


In [ ]:
HEATMAP_COLUMNS = [
    "price_usd",
    "return_24h",
    "volume_24h_usd",
    "market_cap_usd",
    "market_cap_rank",
    "momentum_30d_pct",
    "drawdown_from_peak_pct",
    "news_count_10d",
    "news_delta",
]
SHORT = {
    "price_usd": "Price",
    "return_24h": "24h return",
    "volume_24h_usd": "Dollar volume",
    "market_cap_usd": "Market value",
    "market_cap_rank": "Rank in our 15",
    "momentum_30d_pct": "30-day change",
    "drawdown_from_peak_pct": "Vs 30-day high",
    "news_count_10d": "Headline count",
    "news_delta": "News points",
}
numbers = labeled[HEATMAP_COLUMNS].apply(pd.to_numeric, errors="coerce")
correlation = numbers.corr()
pairs = []
for i, left in enumerate(HEATMAP_COLUMNS):
    for right in HEATMAP_COLUMNS[i + 1 :]:
        value = float(correlation.loc[left, right])
        pairs.append((abs(value), value, left, right))
pairs.sort(reverse=True)
print("Strongest straight-line pairs")
for _, value, left, right in pairs[:6]:
    print(f"{value:+.3f}  {left}  with  {right}")

fig, ax = plt.subplots(figsize=(6.4, 4.5))
image = ax.imshow(correlation.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(HEATMAP_COLUMNS)))
ax.set_yticks(range(len(HEATMAP_COLUMNS)))
ax.set_xticklabels([SHORT[c] for c in HEATMAP_COLUMNS], rotation=40, ha="right", fontsize=8)
ax.set_yticklabels([SHORT[c] for c in HEATMAP_COLUMNS], fontsize=8)
for i in range(len(HEATMAP_COLUMNS)):
    for j in range(len(HEATMAP_COLUMNS)):
        value = correlation.values[i, j]
        if i == j or abs(value) < 0.5:
            continue
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7, color="black")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Straight-line correlation among the numeric columns")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_correlation_heatmap.png", dpi=140, bbox_inches="tight")
plt.show()


## 5. How alike are two rows?

Correlation compares columns. A trading question often compares rows: is this coin-day like that coin-day? Our rows are mixed. They hold dollar amounts, percents, ranks, category names, exchange names, a list of past prices, and headline text. Euclidean distance subtracts every field and squares the gap. On the raw file that subtraction is dominated by market value, because a market value is in the billions and a percent is a small number. We show that with Bitcoin, Ethereum, and Pepe on the same day. The measure that respects mixed fields is Gower distance. Each numeric field is scored by how far apart the two rows are on that field's own range that day, each name either matches or it does not, and every field counts once.


In [ ]:
NUMERIC = HEATMAP_COLUMNS
CATEGORICAL = ["category", "primary_exchange", "press_opinion"]
sample_day = pd.Timestamp("2026-05-24")
day = labeled[labeled["report_day"].eq(sample_day)].set_index("token_symbol")
raw_numbers = day[NUMERIC].apply(pd.to_numeric, errors="coerce")

def euclidean(left, right):
    gap = raw_numbers.loc[left] - raw_numbers.loc[right]
    return float(np.sqrt((gap ** 2).sum())), gap ** 2

btc_pepe, btc_pepe_sq = euclidean("BTC", "PEPE")
btc_eth, _ = euclidean("BTC", "ETH")
mcap_share = float(btc_pepe_sq["market_cap_usd"] / btc_pepe_sq.sum())
print(f"Raw Euclidean distance on {sample_day.date()}")
print(f"Bitcoin to Pepe: {btc_pepe:,.0f}")
print(f"Bitcoin to Ethereum: {btc_eth:,.0f}")
print(f"Share of the Bitcoin–Pepe squared distance that is market value alone: {mcap_share:.2%}")

ranges = raw_numbers.max() - raw_numbers.min()
ranges = ranges.replace(0, np.nan)

def gower(left, right):
    numeric_gaps = ((raw_numbers.loc[left] - raw_numbers.loc[right]).abs() / ranges).dropna()
    name_gaps = [float(day.loc[left, col] != day.loc[right, col]) for col in CATEGORICAL]
    return float(np.mean(list(numeric_gaps.values) + name_gaps))

print("\nSimple Gower distance on the same day (0 = identical, 1 = nothing in common)")
for left, right in [("BTC", "ETH"), ("BTC", "PEPE"), ("DOGE", "PEPE"), ("AAVE", "UNI")]:
    print(f"{left} to {right}: {gower(left, right):.3f}")
print("Same category?", {sym: day.loc[sym, "category"] for sym in ["BTC", "ETH", "DOGE", "PEPE", "AAVE", "UNI"]})


## 6. The category label hides the coin

"Meme" and "DeFi" sound like strategies. They are labels on a handful of tickers, each observed for 34 days. We plot the share of days each ticker was higher 90 days later so a category average cannot hide a split inside the group. Dogecoin and Pepe are the test. Both are meme coins. If the label were the useful fact, they would finish near each other.


In [ ]:
by_token = up_rate_table(labeled, "token_symbol").sort_values("up_rate")
print(by_token.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
perfect = by_token.loc[by_token["up_rate"].eq(1), "token_symbol"].tolist()
print("Higher on every labeled day:", ", ".join(perfect), f"({len(perfect)} coins)")

fig, ax = plt.subplots(figsize=(6.2, 3.5))
colors = []
for symbol in by_token["token_symbol"]:
    if symbol == "DOGE":
        colors.append("#b85c38")
    elif symbol == "PEPE":
        colors.append("#2a6f7f")
    else:
        colors.append("#1f4e79")
ax.barh(by_token["token_symbol"], by_token["up_rate"] * 100, color=colors)
ax.set_xlabel("Share of 34 labeled days that were higher 90 days later")
ax.set_xlim(0, 108)
ax.set_title("Dogecoin and Pepe do not belong in one decision")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_up_rate_by_token.png", dpi=140, bbox_inches="tight")
plt.show()


## 7. Does the dip pattern survive the calendar?

Section 3 showed that weaker 30-day momentum lined up with a higher chance of being up 90 days later. Section 3 also showed that later report dates were much more often up. Those two facts can be the same fact: the market was falling into mid-June, and the 90-day window after mid-June landed in a recovery. We split the labeled rows at June 10 and recompute the momentum groups inside each half. If the pattern vanishes inside both halves, the full-sample chart was mostly the calendar. The last lines round the same counts the memo quotes, so the Word file and this run can be checked against each other.


In [ ]:
labeled["half"] = np.where(labeled["report_day"] < mid, "Before June 10", "June 10 and after")
for half_name, frame in labeled.groupby("half"):
    frame = frame.copy()
    frame["momentum_quartile"] = pd.qcut(pd.to_numeric(frame["momentum_30d_pct"]), 4, duplicates="drop")
    print("\n" + str(half_name))
    print(
        up_rate_table(frame, "momentum_quartile")
        .sort_values("momentum_quartile")
        .to_string(index=False, float_format=lambda v: f"{v:,.4f}")
    )
day_momentum = labeled.groupby("report_day")["momentum_30d_pct"].median()
print("\nCorrelation of 30-day momentum with the report date:", f"{pd.to_numeric(labeled['momentum_30d_pct']).corr(labeled['report_day'].map(pd.Timestamp.toordinal)):+.3f}")
print("Correlation of the 90-day result with the report date:", f"{labeled['positive_90d_return'].corr(labeled['report_day'].map(pd.Timestamp.toordinal)):+.3f}")
print("Correlation of 30-day momentum with the 90-day result:", f"{pd.to_numeric(labeled['momentum_30d_pct']).corr(labeled['positive_90d_return']):+.3f}")

print("\nSame rounded figures the memo quotes")
print(f"File rows {len(raw)}; labeled {len(labeled)}; up {n_up} ({n_up / len(labeled):.1%}); not up {n_down} ({n_down / len(labeled):.1%})")
print(f"Price mean ${price.mean():,.0f}, median ${price.median():.2f}")
volume = pd.to_numeric(labeled['volume_24h_usd'])
print(f"Dollar volume mean ${volume.mean()/1e9:.2f} billion, median ${volume.median()/1e6:.0f} million")
print(f"Market value mean ${mcap.mean()/1e9:.1f} billion, median ${mcap.median()/1e9:.2f} billion")
print(f"Bitcoin plus Ethereum share of summed prices {btc_eth_price_share:.1%}; Bitcoin share of summed market value {btc_mcap_share:.1%}")
print(f"Rows with no headline {zero_news_share:.1%}")
print(f"Before June 10: {int(early.sum())} of {len(early)}. On or after June 10: {int(late.sum())} of {len(late)}.")
